In [0]:
%sql
USE CATALOG education_lakehouse;

In [0]:
%sql
SELECT
    current_catalog() AS catalog_name,
    current_schema() AS schema_name;

In [0]:
%sql
USE SCHEMA bronze;

In [0]:
%sql
SELECT 
    current_catalog() AS current_catalog,
    current_schema() AS current_schema;

In [0]:
import os

# Check current execution directory
print("Current working directory:")
print(os.getcwd())

# List available files
print("\nFiles and folders:")
print(os.listdir("."))

In [0]:
import os

project_path = os.path.abspath("..")

data_path = os.path.join(
    project_path,
    "data",
    "sample"
)

print("Project path:", project_path)
print("Data path:", data_path)

print("Directory exists:", os.path.exists(data_path))
print("Files:", os.listdir(data_path))

In [0]:
import pandas as pd
import os

files = [
    "applications.csv",
    "courses.csv",
    "students.csv",
    "universities.csv"
]
for file in files:
    file_path = os.path.join(data_path,file)

    df = pd.read_csv(file_path)
    print(f"Records: {len(df)}")
    print(f"Columns: {list(df.columns)}")

In [0]:
from pyspark.sql import functions as F
#Reading the data 
applications_path = os.path.join(
    data_path,
    "applications.csv"
)
applications_df = (
    spark.read
    .format("csv")
    .option("header","true")
    .option("inferSchema","false")
    .load(applications_path)
)
applications_df.printSchema()
display(applications_df.limit(2))
print("Total records:", applications_df.count())

In [0]:
#Adding Ingestion Metadata
from pyspark.sql.functions import (
    current_timestamp,
    input_file_name,
    col,
    current_date
)
applications_bronze_df = (
    applications_df
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )
    .withColumn(
        "ingestion_date",
        current_date()
    )
)
applications_bronze_df.printSchema()
display(applications_bronze_df.limit(10))
#validation checks
print("Total records:", applications_bronze_df.count())

print(
    "Distinct application IDs:",
    applications_bronze_df.select("application_id").distinct().count()
)

print(
    "Null application IDs:",
    applications_bronze_df.filter(
        F.col("application_id").isNull()
    ).count()
)

In [0]:
duplicate_applications = (
    applications_bronze_df
    .groupBy("application_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_applications)
null_applications = (
    applications_bronze_df
    .filter(F.col("application_id").isNull())
)

display(null_applications)

In [0]:
#write dataframe to save as delta table 
(
    applications_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "education_lakehouse.bronze.bronze_applications"
    )
)
saved_df = spark.table(
    "education_lakehouse.bronze.bronze_applications"
)

print("Bronze record count:", saved_df.count())
display(saved_df.limit(10))

In [0]:
from pyspark.sql import functions as F

courses_path = os.path.join(
    data_path,
    "courses.csv"
)

#Reading the data 
courses_df = (
    spark.read
    .format("csv")
    .option("header","true")
    .option("inferSchema","false")
    .load(courses_path)
)
courses_df.printSchema()